# Sliding Window for LSTM

This notebook explains how to transform a time-series sequence into input-output samples for an LSTM.

Example sequence:

```text
[10, 20, 30, 40, 50, 60, 70]
```

With a window size of `3`, the samples become:

```text
{input: [10, 20, 30], output: [40]}
{input: [20, 30, 40], output: [50]}
{input: [30, 40, 50], output: [60]}
{input: [40, 50, 60], output: [70]}
```

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd

## 2. Create the time-series data

In [ ]:
data = [10, 20, 30, 40, 50, 60, 70]

print("Original data:", data)

## 3. Understand one sliding window

For a window size of `3`:

- The first three values become the input.
- The next value becomes the output.

In [ ]:
window_size = 3

first_input = data[0:3]
first_output = data[3]

print({
    "input": first_input,
    "output": [first_output]
})

## 4. Create the sliding-window function

The window moves forward by one position on every loop.

In [ ]:
def create_sliding_windows(data, window_size):
    samples = []

    for i in range(len(data) - window_size):
        input_window = data[i:i + window_size]
        output_value = data[i + window_size]

        sample = {
            "input": input_window,
            "output": [output_value]
        }

        samples.append(sample)

    return samples

## 5. Display samples in `{input, output}` format

In [ ]:
samples = create_sliding_windows(data, window_size=3)

for sample in samples:
    print(sample)

The expected result is:

```text
{'input': [10, 20, 30], 'output': [40]}
{'input': [20, 30, 40], 'output': [50]}
{'input': [30, 40, 50], 'output': [60]}
{'input': [40, 50, 60], 'output': [70]}
```

## 6. Create `X` and `y` for LSTM training

The dictionary format is useful for understanding the windows. For model training, we separate the samples into `X` and `y`.

In [ ]:
def create_sequences(data, window_size):
    X = []
    y = []

    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])

    return np.array(X), np.array(y)

In [ ]:
X, y = create_sequences(data, window_size=3)

print("X:")
print(X)

print("\ny:")
print(y)

## 7. Check the current shapes

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

Currently, `X` has shape:

```text
(4, 3)
```

This means:

- `4` samples
- `3` timesteps in every sample

However, an LSTM expects three-dimensional input:

```text
(samples, timesteps, features)
```

## 8. Reshape the data for LSTM

In [ ]:
X = X.reshape(X.shape[0], X.shape[1], 1)

print("LSTM input shape:", X.shape)

The new shape is:

```text
(4, 3, 1)
```

This means:

- `4` training samples
- `3` timesteps in every sample
- `1` feature at every timestep

## 9. Display every reshaped LSTM sample

In [ ]:
for i in range(len(X)):
    print(f"Sample {i + 1}")
    print("Input:", X[i].flatten().tolist())
    print("Output:", [int(y[i])])
    print()

## 10. Install TensorFlow if required

Uncomment and run the following cell only if TensorFlow is not already installed.

In [ ]:
# !pip install tensorflow

## 11. Import TensorFlow/Keras

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense

## 12. Build the LSTM model

`Input(shape=(3, 1))` means:

- Three previous timesteps are provided.
- Each timestep contains one feature.

In [ ]:
model = Sequential([
    Input(shape=(window_size, 1)),
    LSTM(32),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

## 13. Train the LSTM

This dataset is intentionally small for understanding the sliding-window process.

In [ ]:
history = model.fit(
    X,
    y,
    epochs=200,
    verbose=0
)

print("Model training completed.")

## 14. Predict the next value

To predict the value after `70`, use the latest three values:

```text
[50, 60, 70]
```

In [ ]:
latest_window = np.array([50, 60, 70], dtype=np.float32)

latest_window = latest_window.reshape(1, window_size, 1)

next_prediction = model.predict(latest_window, verbose=0)

print("Input window:", [50, 60, 70])
print("Predicted next value:", float(next_prediction[0][0]))

## 15. Complete result in input-output format

In [ ]:
result = {
    "input": [50, 60, 70],
    "output": [round(float(next_prediction[0][0]), 2)]
}

print(result)

## 16. Important concept

The sliding window is a **data-preparation technique**. It is not an internal LSTM layer.

```text
Original sequence
        ↓
Sliding-window samples
        ↓
(samples, timesteps, features)
        ↓
LSTM model
        ↓
Next-value prediction
```

For this example:

```text
X shape = (4, 3, 1)

4 = number of samples
3 = number of previous timesteps
1 = number of features per timestep
```

## 17. Final verification

In [ ]:
assert X.shape == (4, 3, 1)
assert y.shape == (4,)
assert samples[0] == {"input": [10, 20, 30], "output": [40]}
assert samples[-1] == {"input": [40, 50, 60], "output": [70]}
assert latest_window.shape == (1, 3, 1)

print("All sliding-window tests passed.")